# E074 -- Dark Energy: The Accelerating Universe (Pantheon+)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaulVanCode/protoscience-nasa-experiments/blob/main/notebooks/e074_dark_energy.ipynb)

**Objective:** Download the Pantheon+ Type Ia supernova compilation, fit the LCDM cosmological model to the Hubble diagram, scan the matter density parameter Omega_m, and demonstrate that the expansion of the universe is accelerating.

Type Ia supernovae are "standard candles" — their intrinsic luminosity is known, so measuring their apparent brightness gives us their distance. Comparing distance vs redshift for thousands of supernovae reveals that the expansion is speeding up, implying the existence of dark energy (~68% of the universe's energy content).

## Data Source

- **Dataset:** Pantheon+ SH0ES Type Ia Supernova Compilation
- **Reference:** Scolnic et al. (2022), Brout et al. (2022)
- **URL:** GitHub PantheonPlusSH0ES DataRelease
- **Columns used:** zHD (CMB-frame redshift), MU_SH0ES (distance modulus)
- **License:** Public (academic)

In [ ]:
import urllib.request
import numpy as np
from scipy import stats
from scipy.integrate import quad
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt

# Download Pantheon+ data
url = ("https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
       "Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat")
print("Downloading Pantheon+ SH0ES supernova data...")
req = urllib.request.Request(url, headers={'User-Agent': 'ProtoScience/1.0'})
response = urllib.request.urlopen(req, timeout=120)
raw = response.read().decode('utf-8')
lines = raw.strip().split('\n')
print(f"Downloaded {len(lines)} lines")
print(f"Header: {lines[0][:120]}")

In [ ]:
# Parse space-separated data
header = lines[0].split()
col_idx = {h: i for i, h in enumerate(header)}
print(f"Columns: {header}")

# Find key columns
z_col = 'zHD'
mu_col = 'MU_SH0ES'
mu_err_col = 'MU_SH0ES_ERR_DIAG' if 'MU_SH0ES_ERR_DIAG' in col_idx else None

z_data = []
mu_data = []
mu_err = []
sn_names = []

for line in lines[1:]:
    parts = line.split()
    if len(parts) < len(header):
        continue
    try:
        z = float(parts[col_idx[z_col]])
        mu = float(parts[col_idx[mu_col]])
        if z > 0.001 and mu > 0:
            z_data.append(z)
            mu_data.append(mu)
            if mu_err_col:
                mu_err.append(float(parts[col_idx[mu_err_col]]))
            else:
                mu_err.append(0.1)  # default
            sn_names.append(parts[col_idx.get('CID', col_idx.get('SN', 0))])
    except (ValueError, IndexError):
        continue

z_data = np.array(z_data)
mu_data = np.array(mu_data)
mu_err = np.array(mu_err)

print(f"\nParsed {len(z_data)} Type Ia supernovae")
print(f"Redshift range: [{z_data.min():.4f}, {z_data.max():.4f}]")
print(f"Distance modulus range: [{mu_data.min():.2f}, {mu_data.max():.2f}] mag")

## LCDM Cosmological Model

In the flat LCDM model, the luminosity distance is:

$$d_L(z) = \frac{c(1+z)}{H_0} \int_0^z \frac{dz'}{\sqrt{\Omega_m(1+z')^3 + (1-\Omega_m)}}$$

The distance modulus is $\mu = 5 \log_{10}(d_L / 10\,\text{pc})$. We fit for $\Omega_m$ while marginalizing over $H_0$ (absorbed into an additive constant in $\mu$).

In [ ]:
# LCDM luminosity distance
c_kms = 299792.458  # km/s

def E_inv(z, Om):
    """1/E(z) for flat LCDM."""
    return 1.0 / np.sqrt(Om * (1 + z)**3 + (1 - Om))

def mu_model(z, Om, M_offset):
    """Distance modulus for flat LCDM. M_offset absorbs H0 + absolute magnitude."""
    mu_vals = np.zeros_like(z)
    for i, zi in enumerate(z):
        integral, _ = quad(E_inv, 0, zi, args=(Om,))
        dL = (1 + zi) * integral  # in units of c/H0
        mu_vals[i] = 5.0 * np.log10(dL) + M_offset
    return mu_vals

def chi2(Om, z, mu_obs, mu_e):
    """Chi-squared for LCDM fit, analytically marginalizing over M_offset."""
    # Compute model mu up to an additive constant
    mu_mod = np.zeros_like(z)
    for i, zi in enumerate(z):
        integral, _ = quad(E_inv, 0, zi, args=(Om,))
        dL = (1 + zi) * integral
        mu_mod[i] = 5.0 * np.log10(max(dL, 1e-30))
    
    # Analytically marginalize over offset
    w = 1.0 / mu_e**2
    delta = mu_obs - mu_mod
    offset_best = np.sum(w * delta) / np.sum(w)
    return np.sum(w * (delta - offset_best)**2)

# Omega_m scan
Om_grid = np.linspace(0.05, 0.60, 50)
chi2_grid = np.array([chi2(Om, z_data, mu_data, mu_err) for Om in Om_grid])

# Find best fit
result = minimize_scalar(lambda Om: chi2(Om, z_data, mu_data, mu_err),
                         bounds=(0.05, 0.60), method='bounded')
Om_best = result.x

# Also fit a simple linear (empty universe) model for comparison
# mu_linear = 5*log10(z) + const
log_z = np.log10(z_data)
slope_lin, intercept_lin, _, _, _ = stats.linregress(log_z, mu_data)

print(f"=== LCDM Fit Results ===")
print(f"  Best-fit Omega_m = {Om_best:.3f}")
print(f"  Omega_Lambda = {1 - Om_best:.3f}")
print(f"  Planck value: Omega_m = 0.315 ± 0.007")
print(f"  chi2/dof = {result.fun:.1f}/{len(z_data)-2} = {result.fun/(len(z_data)-2):.3f}")
print(f"\n  Linear model slope: {slope_lin:.3f} (expect 5.0 for Hubble law)")

In [ ]:
# Compute best-fit model curve
z_model = np.logspace(np.log10(z_data.min()), np.log10(z_data.max()), 100)

# Best-fit LCDM
mu_lcdm = mu_model(z_model, Om_best, 0)
# Find offset
mu_data_model = mu_model(z_data, Om_best, 0)
w = 1.0 / mu_err**2
offset = np.sum(w * (mu_data - mu_data_model)) / np.sum(w)
mu_lcdm += offset

# Empty universe (Om=0)
mu_empty = mu_model(z_model, 0.001, 0)
mu_data_empty = mu_model(z_data, 0.001, 0)
offset_empty = np.sum(w * (mu_data - mu_data_empty)) / np.sum(w)
mu_empty += offset_empty

# Matter-only (Om=1)
mu_matter = mu_model(z_model, 1.0, 0)
mu_data_matter = mu_model(z_data, 1.0, 0)
offset_matter = np.sum(w * (mu_data - mu_data_matter)) / np.sum(w)
mu_matter += offset_matter

# === 4-SUBPLOT FIGURE ===
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle("E074: Dark Energy — The Accelerating Universe (Pantheon+)",
             fontsize=15, fontweight='bold')

# (a) Hubble diagram with models
ax = axes[0, 0]
ax.scatter(z_data, mu_data, s=2, alpha=0.2, color='gray', label=f'Pantheon+ ({len(z_data)} SNe)')
ax.plot(z_model, mu_lcdm, 'r-', linewidth=2.5,
        label=f'ΛCDM (Ω$_m$={Om_best:.3f})')
ax.plot(z_model, mu_empty, 'b--', linewidth=1.5, alpha=0.7, label='Empty (Ω$_m$=0)')
ax.plot(z_model, mu_matter, 'g--', linewidth=1.5, alpha=0.7, label='Matter only (Ω$_m$=1)')
ax.set_xscale('log')
ax.set_xlabel('Redshift z', fontsize=11)
ax.set_ylabel('Distance Modulus μ [mag]', fontsize=11)
ax.set_title('(a) Supernova Hubble Diagram', fontsize=12)
ax.legend(fontsize=9)

# (b) Residuals from LCDM
ax = axes[0, 1]
mu_data_best = mu_model(z_data, Om_best, 0) + offset
residuals = mu_data - mu_data_best
ax.scatter(z_data, residuals, s=2, alpha=0.2, color='steelblue')
ax.axhline(0, color='red', linewidth=1.5)
# Bin residuals
z_edges = np.logspace(np.log10(z_data.min()), np.log10(z_data.max()), 20)
z_mids, res_meds, res_errs = [], [], []
for i in range(len(z_edges) - 1):
    m = (z_data >= z_edges[i]) & (z_data < z_edges[i + 1])
    if m.sum() > 5:
        z_mids.append(np.sqrt(z_edges[i] * z_edges[i + 1]))
        res_meds.append(np.median(residuals[m]))
        res_errs.append(np.std(residuals[m]) / np.sqrt(m.sum()))
ax.errorbar(z_mids, res_meds, yerr=res_errs, fmt='ro', markersize=5, capsize=3)
ax.set_xscale('log')
ax.set_xlabel('Redshift z', fontsize=11)
ax.set_ylabel('Residual Δμ [mag]', fontsize=11)
ax.set_title('(b) Residuals from Best-fit ΛCDM', fontsize=12)
ax.set_ylim(-1, 1)

# (c) Omega_m scan (chi2)
ax = axes[1, 0]
delta_chi2 = chi2_grid - chi2_grid.min()
ax.plot(Om_grid, delta_chi2, 'steelblue', linewidth=2)
ax.axvline(Om_best, color='red', linestyle='--', linewidth=2,
           label=f'Best fit: Ω$_m$ = {Om_best:.3f}')
ax.axvline(0.315, color='green', linestyle=':', linewidth=2,
           label='Planck: Ω$_m$ = 0.315')
ax.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
ax.axhline(4.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Ω$_m$', fontsize=11)
ax.set_ylabel('Δχ²', fontsize=11)
ax.set_title('(c) Ω$_m$ Likelihood Scan', fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, min(30, delta_chi2.max()))

# (d) Redshift distribution of supernovae
ax = axes[1, 1]
ax.hist(z_data, bins=50, color='coral', edgecolor='black', linewidth=0.3, alpha=0.8)
ax.set_xlabel('Redshift z', fontsize=11)
ax.set_ylabel('Number of Supernovae', fontsize=11)
ax.set_title(f'(d) Pantheon+ Redshift Distribution (N={len(z_data)})', fontsize=12)
ax.annotate(f'Median z = {np.median(z_data):.3f}\n'
            f'Max z = {z_data.max():.3f}',
            xy=(0.65, 0.8), xycoords='axes fraction', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('e074_dark_energy.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved: e074_dark_energy.png")

## Key Results Summary

| Analysis | Result |
|----------|--------|
| Supernovae in sample | Pantheon+ compilation (1000+) |
| Best-fit Omega_m | ~0.3 (consistent with Planck 0.315) |
| Omega_Lambda | ~0.7 (dark energy fraction) |
| LCDM vs empty universe | LCDM strongly preferred |
| Acceleration evidence | Supernovae at z~0.5 are fainter than matter-only prediction |

**Physical interpretation:** Distant Type Ia supernovae are systematically fainter than expected in a decelerating universe — the expansion is accelerating. This discovery (Riess et al. 1998, Perlmutter et al. 1999, Nobel Prize 2011) implies that ~68% of the universe's energy content is "dark energy" driving the acceleration. The Pantheon+ dataset with 1000+ supernovae pins down Omega_m to ~0.3, completing the cosmic energy budget: 5% baryons, 27% dark matter, 68% dark energy.